# 🚀 DMAE-VAD: Decoupled Dual-Stream Masked Autoencoder for Video Anomaly Detection
### Train from Scratch on Dual GPUs (2x Tesla T4) — No Warm-Start Dependency
**Tác giả:** Huỳnh Quang Vũ (MSSV: 2351010247)  
**GVHD:** TS. Lê Viết Tuấn  
**Đơn vị:** Khoa Công nghệ Thông tin - Trường Đại học Mở TP.HCM

## ⚙️ PHẦN 1: HUẤN LUYỆN TRÊN TẬP DỮ LIỆU UCF-CRIME

In [ ]:
# ========================= USER CONFIG: UCF-CRIME =========================
DATASET = "ucf"
MAX_EPOCHS = 15
UCF_BATCH_SIZE = 64        # 32 samples per GPU on 2x T4
LR = 1e-4
NUM_WORKERS = 2
SEED = 234
NUM_TCN_LAYERS = 2
NUM_MAE_LAYERS = 2
MASK_RATIO = 0.5
LOSS_MAE_WEIGHT = 1.0
LOSS_REC_CONTRAST_WEIGHT = 0.5

# Đường dẫn GitHub repository (hoặc upload folder DMAE-VAD trực tiếp vào Kaggle)
REPO_URL = "https://github.com/Sharvuz/DMAE-VAD.git"   # Thay bằng URL repo của bạn nếu có
USE_LOCAL_DIR = False

In [ ]:
# ========================= ENVIRONMENT SETUP =========================
import os, sys, shutil, subprocess, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "DMAE-VAD"
ARTIFACTS = WORK / "dmae_artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print("📦 Đang chuẩn bị môi trường...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "pandas", "tqdm"], check=True)

import torch
print("\n--- THÔNG TIN HỆ THỐNG KAGGLE ---")
print("PyTorch Version:", torch.__version__)
print("CUDA Runtime:", torch.version.cuda)
gpu_count = torch.cuda.device_count()
print(f"Số lượng GPU khả dụng: {gpu_count}")
for i in range(gpu_count):
    print(f"  [GPU {i}]: {torch.cuda.get_device_name(i)}")
assert gpu_count > 0, "⚠️ Vui lòng bật GPU Accelerator (T4 x 2) trong phần Notebook Settings!"
if gpu_count > 1:
    print("🔥 Đã phát hiện 2x GPUs! Chế độ DataParallel sẽ tự động kích hoạt.")

# Chuẩn bị mã nguồn
if not REPO.exists():
    if not USE_LOCAL_DIR and REPO_URL:
        print(f"📥 Đang clone repository từ {REPO_URL}...")
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
        except Exception as e:
            print(f"Clone git chưa thành công ({e}). Tìm kiếm trong /kaggle/input...")
    if not REPO.exists():
        found_code = list(INPUT.rglob("model.py"))
        if found_code:
            src_parent = found_code[0].parent.parent
            print(f"Tìm thấy mã nguồn tại {src_parent}, đang sao chép vào {REPO}...")
            shutil.copytree(src_parent, REPO)
        else:
            print("⚠️ Lưu ý: Bạn có thể tải thư mục DMAE-VAD lên Kaggle Dataset để notebook tự nhận.")

print("✅ Môi trường đã sẵn sàng!")

In [ ]:
# ========================= SMART DATASET PATH RESOLVER =========================
from collections import defaultdict
from pathlib import PurePosixPath
import pandas as pd
import numpy as np

if DATASET == "ucf":
    csv_specs = [
        ("list/ucf_CLIP_rgb.csv", "ucf_train.csv"),
        ("list/ucf_CLIP_rgbtest.csv", "ucf_test.csv"),
    ]
    zip_hints = ["ucfclipfeatures", "ucf_clip"]
else:
    csv_specs = [
        ("list/xd_CLIP_rgb.csv", "xd_train.csv"),
        ("list/xd_CLIP_rgbtest.csv", "xd_test.csv"),
    ]
    zip_hints = ["xdtrainclipfeatures", "xdtestclipfeatures", "xd_clip"]

def basename(path_string):
    return PurePosixPath(str(path_string).replace("\\", "/")).name

required_names = set()
for csv_rel, _ in csv_specs:
    csv_full = REPO / csv_rel
    if csv_full.exists():
        frame = pd.read_csv(csv_full)
        required_names.update(basename(p) for p in frame["path"])

feature_roots = [INPUT]
def indexed_names(roots):
    return {p.name for root in roots for p in root.rglob("*.npy")}

missing_before = required_names - indexed_names(feature_roots)
if missing_before:
    extract_root = WORK / f"dmae_features_{DATASET}"
    extract_root.mkdir(parents=True, exist_ok=True)
    archives = [
        p for p in INPUT.rglob("*.zip")
        if any(hint in p.stem.lower() for hint in zip_hints)
    ]
    for archive_path in archives:
        dest = (extract_root / archive_path.stem).resolve()
        dest.mkdir(parents=True, exist_ok=True)
        print(f"📦 Đang giải nén {archive_path.name}...")
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(dest)
    feature_roots.append(extract_root)

index = defaultdict(list)
for root in feature_roots:
    for path in root.rglob("*.npy"):
        if path.name in required_names:
            index[path.name].append(path)

def rewrite_csv(csv_rel, output_name):
    frame = pd.read_csv(REPO / csv_rel)
    resolved = []
    for original in frame["path"].astype(str):
        b_name = basename(original)
        matches = index.get(b_name, [])
        if matches:
            resolved.append(str(matches[0]))
        else:
            resolved.append(original)
    output = ARTIFACTS / output_name
    frame["path"] = resolved
    frame.to_csv(output, index=False)
    print(f"✅ Đã đồng bộ {output_name} với {len(frame)} dòng!")
    return output

train_csv = rewrite_csv(*csv_specs[0])
test_csv = rewrite_csv(*csv_specs[1])
print("Dữ liệu đã sẵn sàng để huấn luyện.")

In [ ]:
# ========================= TRAIN DMAE-VAD (UCF-CRIME) =========================
import subprocess, sys, time
from pathlib import Path

dataset_dir = ARTIFACTS / "ucf"
dataset_dir.mkdir(parents=True, exist_ok=True)

train_cmd = [
    sys.executable, "-u", "src/ucf_train.py",
    "--dataset", "ucf",
    "--train-list", str(ARTIFACTS / "ucf_train.csv"),
    "--test-list", str(ARTIFACTS / "ucf_test.csv"),
    "--gt-path", str(REPO / "list/gt_ucf.npy"),
    "--gt-segment-path", str(REPO / "list/gt_segment_ucf.npy"),
    "--gt-label-path", str(REPO / "list/gt_label_ucf.npy"),
    "--save-dir", str(dataset_dir),
    
    # Siêu tham số huấn luyện từ đầu (End-to-End)
    "--max-epoch", str(MAX_EPOCHS),
    "--batch-size", str(UCF_BATCH_SIZE),
    "--lr", str(LR),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
    
    # Kiến trúc DMAE-VAD
    "--num-tcn-layers", str(NUM_TCN_LAYERS),
    "--num-mae-layers", str(NUM_MAE_LAYERS),
    "--mask-ratio", str(MASK_RATIO),
    "--loss-mae-weight", str(LOSS_MAE_WEIGHT),
    "--loss-rec-contrast-weight", str(LOSS_REC_CONTRAST_WEIGHT)
]

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN DMAE-VAD TỪ CON SỐ 0 TRÊN UCF-CRIME ({MAX_EPOCHS} Epochs)...\n", flush=True)
started = time.time()
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    train_cmd, cwd=str(REPO), env=env, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1
)

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"❌ Huấn luyện gặp sự cố với mã lỗi {return_code}")
print(f"\n✅ HOÀN THÀNH HUẤN LUYỆN UCF-CRIME TRONG {(time.time() - started)/60:.2f} PHÚT!")

In [ ]:
# ========================= EVALUATION: UCF-CRIME =========================
best_model_path = ARTIFACTS / "ucf" / "dmae_vad_ucf_best.pth"

eval_cmd = [
    sys.executable, "-u", "src/ucf_test.py",
    "--dataset", "ucf",
    "--test-list", str(ARTIFACTS / "ucf_test.csv"),
    "--gt-path", str(REPO / "list/gt_ucf.npy"),
    "--gt-segment-path", str(REPO / "list/gt_segment_ucf.npy"),
    "--gt-label-path", str(REPO / "list/gt_label_ucf.npy"),
    "--model-path", str(best_model_path),
    "--num-tcn-layers", str(NUM_TCN_LAYERS),
    "--num-mae-layers", str(NUM_MAE_LAYERS)
]

print("🔍 Đang đánh giá lại model tốt nhất trên tập test (Deterministic Dual-Half Inpainting)...")
process_eval = subprocess.Popen(eval_cmd, cwd=str(REPO), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in process_eval.stdout:
    print(line, end="")
    sys.stdout.flush()
process_eval.wait()

## ⚙️ PHẦN 2: HUẤN LUYỆN TRÊN TẬP DỮ LIỆU XD-VIOLENCE

In [ ]:
# ========================= USER CONFIG: XD-VIOLENCE =========================
DATASET_XD = "xd"
MAX_EPOCHS_XD = 20
XD_BATCH_SIZE = 64
LR_XD = 1e-4

dataset_xd_dir = ARTIFACTS / "xd"
dataset_xd_dir.mkdir(parents=True, exist_ok=True)

train_xd_cmd = [
    sys.executable, "-u", "src/xd_train.py",
    "--dataset", "xd",
    "--train-list", str(ARTIFACTS / "xd_train.csv"),
    "--test-list", str(ARTIFACTS / "xd_test.csv"),
    "--gt-path", str(REPO / "list/gt.npy"),
    "--gt-segment-path", str(REPO / "list/gt_segment.npy"),
    "--gt-label-path", str(REPO / "list/gt_label.npy"),
    "--save-dir", str(dataset_xd_dir),
    
    "--max-epoch", str(MAX_EPOCHS_XD),
    "--batch-size", str(XD_BATCH_SIZE),
    "--lr", str(LR_XD),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
    
    "--num-tcn-layers", str(NUM_TCN_LAYERS),
    "--num-mae-layers", str(NUM_MAE_LAYERS),
    "--mask-ratio", str(MASK_RATIO),
    "--loss-mae-weight", str(LOSS_MAE_WEIGHT),
    "--loss-rec-contrast-weight", str(LOSS_REC_CONTRAST_WEIGHT)
]

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN DMAE-VAD TỪ CON SỐ 0 TRÊN XD-VIOLENCE ({MAX_EPOCHS_XD} Epochs)...\n", flush=True)
started = time.time()

process_xd = subprocess.Popen(
    train_xd_cmd, cwd=str(REPO), env=env, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1
)

for line in process_xd.stdout:
    print(line, end="")
    sys.stdout.flush()

return_code_xd = process_xd.wait()
if return_code_xd != 0:
    raise RuntimeError(f"❌ Huấn luyện XD thất bại với mã lỗi {return_code_xd}")
print(f"\n✅ HOÀN THÀNH HUẤN LUYỆN XD-VIOLENCE TRONG {(time.time() - started)/60:.2f} PHÚT!")

In [ ]:
# ========================= EVALUATION: XD-VIOLENCE =========================
best_model_xd = ARTIFACTS / "xd" / "dmae_vad_xd_best.pth"

eval_xd_cmd = [
    sys.executable, "-u", "src/xd_test.py",
    "--dataset", "xd",
    "--test-list", str(ARTIFACTS / "xd_test.csv"),
    "--gt-path", str(REPO / "list/gt.npy"),
    "--gt-segment-path", str(REPO / "list/gt_segment.npy"),
    "--gt-label-path", str(REPO / "list/gt_label.npy"),
    "--model-path", str(best_model_xd),
    "--num-tcn-layers", str(NUM_TCN_LAYERS),
    "--num-mae-layers", str(NUM_MAE_LAYERS)
]

print("🔍 Đang đánh giá lại model XD tốt nhất trên tập test...")
process_eval_xd = subprocess.Popen(eval_xd_cmd, cwd=str(REPO), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in process_eval_xd.stdout:
    print(line, end="")
    sys.stdout.flush()
process_eval_xd.wait()